In [ ]:
# ============================================================
# HPV E6/E7/L1 Conservation + Shannon Entropy Heatmaps
#
# Features:
# 1. Human / Feline / Combined heatmaps
# 2. Conservation
# 3. Shannon entropy
# 4. Viridis blue -> green -> yellow color scale
# 5. X-axis based on a USER-SELECTED REFERENCE SEQUENCE


# ============================================================

import os
import csv
import math
from collections import Counter
from html import escape


# ============================================================
# 1. SETTINGS
# ============================================================

# ------------------------------------------------------------
# Alignment file
# ------------------------------------------------------------

alignment_file = r"E:\..................\L1-aligned.fasta"

# DNA or PROTEIN
sequence_type = "DNA"

# Output folder
output_dir = r"E:\............\conservation_heatmaps-L1"

os.makedirs(output_dir, exist_ok=True)


# ============================================================
# 2. SELECT REFERENCE SEQUENCES
# ============================================================

# ============================================================

human_reference = "HPV..."

feline_reference = "..."

combined_reference = "HPV..."


# ============================================================
# 3. READ FASTA
# ============================================================

def read_fasta(path):

    names = []
    sequences = []

    current_name = None
    current_sequence = []

    with open(path, "r", encoding="utf-8-sig") as f:

        for line in f:

            line = line.strip()

            if not line:
                continue

            if line.startswith(">"):

                if current_name is not None:

                    sequences.append(
                        "".join(current_sequence).upper()
                    )

                current_name = line[1:].split()[0]

                names.append(current_name)

                current_sequence = []

            else:

                current_sequence.append(line)

        if current_name is not None:

            sequences.append(
                "".join(current_sequence).upper()
            )

    if len(names) != len(sequences):

        raise ValueError(
            "FASTA parsing error: "
            "number of names and sequences differ."
        )

    return names, sequences


names, sequences = read_fasta(alignment_file)


# ============================================================
# 4. CHECK ALIGNMENT
# ============================================================

if len(sequences) == 0:

    raise ValueError(
        "No sequences found in FASTA file."
    )


alignment_lengths = set(
    len(seq) for seq in sequences
)


if len(alignment_lengths) != 1:

    raise ValueError(
        "Sequences do not have the same alignment length.\n"
        f"Detected lengths: {sorted(alignment_lengths)}"
    )


alignment_length = len(sequences[0])


print("==============================================")
print("ALIGNMENT INFORMATION")
print("==============================================")

print(
    f"Number of sequences : {len(sequences)}"
)

print(
    f"Alignment columns   : {alignment_length}"
)

print(
    f"Sequence type       : {sequence_type}"
)


# ============================================================
# 5. HUMAN / FELINE CLASSIFICATION
# ============================================================

human_sequences = []
feline_sequences = []

human_names = []
feline_names = []


for name, seq in zip(names, sequences):

    if name.upper().startswith("HPV"):

        human_names.append(name)
        human_sequences.append(seq)

    else:

        feline_names.append(name)
        feline_sequences.append(seq)


combined_sequences = sequences


print("\n==============================================")
print("GROUPS")
print("==============================================")

print(
    f"Human sequences  : {len(human_sequences)}"
)

print(
    f"Feline sequences : {len(feline_sequences)}"
)

print(
    f"Combined         : {len(combined_sequences)}"
)


if len(human_sequences) == 0:

    raise ValueError(
        "No human sequences found. "
        "Human IDs must start with 'HPV'."
    )


if len(feline_sequences) == 0:

    raise ValueError(
        "No feline sequences found."
    )


# ============================================================
# 6. MISSING / GAP CHARACTERS
# ============================================================

sequence_type = sequence_type.upper()


if sequence_type == "DNA":

    missing_chars = set("-?.N")

    max_entropy = math.log2(4)


elif sequence_type == "PROTEIN":

    missing_chars = set("-?.X")

    max_entropy = math.log2(20)


else:

    raise ValueError(
        "sequence_type must be 'DNA' or 'PROTEIN'"
    )


# ============================================================
# 7. FIND REFERENCE SEQUENCE
# ============================================================

def find_reference(
        reference_keyword,
        group_names,
        group_sequences
    ):

    keyword = reference_keyword.upper()

    matches = []

    for name, seq in zip(
        group_names,
        group_sequences
    ):

        if keyword in name.upper():

            matches.append(
                (name, seq)
            )

    if len(matches) == 0:

        raise ValueError(
            f"\nReference '{reference_keyword}' "
            f"was not found.\n\n"
            f"Available sequences include:\n"
            + "\n".join(group_names[:30])
        )

    if len(matches) > 1:

        print(
            f"\nWARNING: Reference '{reference_keyword}' "
            f"matched {len(matches)} sequences:"
        )

        for name, _ in matches:

            print(
                "   ",
                name
            )

        print(
            "\nThe first matching sequence "
            "will be used."
        )

    reference_name, reference_sequence = matches[0]

    print(
        f"\nReference selected: "
        f"{reference_name}"
    )

    return reference_name, reference_sequence


# ------------------------------------------------------------
# Human reference
# ------------------------------------------------------------

human_reference_name, human_reference_sequence = \
    find_reference(
        human_reference,
        human_names,
        human_sequences
    )


# ------------------------------------------------------------
# Feline reference
# ------------------------------------------------------------

feline_reference_name, feline_reference_sequence = \
    find_reference(
        feline_reference,
        feline_names,
        feline_sequences
    )


# ------------------------------------------------------------
# Combined reference
#
# Search across ALL sequences because Combined contains
# both human and feline sequences.
# ------------------------------------------------------------

combined_reference_name, combined_reference_sequence = \
    find_reference(
        combined_reference,
        names,
        combined_sequences
    )


# ============================================================
# 8. CALCULATE CONSERVATION + SHANNON ENTROPY
# ============================================================

def calculate_metrics(group_sequences):

    conservation = []
    entropy = []
    consensus = []
    valid_count = []

    for alignment_position in range(
        alignment_length
    ):

        column = []

        for seq in group_sequences:

            residue = seq[
                alignment_position
            ].upper()

            if residue not in missing_chars:

                column.append(residue)

        # No valid residues
        if len(column) == 0:

            conservation.append(None)
            entropy.append(None)
            consensus.append("-")
            valid_count.append(0)

            continue

        counts = Counter(column)

        n = len(column)

        # ----------------------------------------------------
        # Conservation
        # ----------------------------------------------------

        most_common_residue, \
        most_common_count = \
            counts.most_common(1)[0]

        cons = (
            most_common_count / n
        )

        conservation.append(cons)

        consensus.append(
            most_common_residue
        )

        valid_count.append(n)

        # ----------------------------------------------------
        # Shannon entropy
        # ----------------------------------------------------

        H = 0.0

        for count in counts.values():

            p = count / n

            H -= p * math.log2(p)

        entropy.append(H)

    return {

        "conservation": conservation,

        "entropy": entropy,

        "consensus": consensus,

        "valid_count": valid_count

    }


human_metrics = calculate_metrics(
    human_sequences
)

feline_metrics = calculate_metrics(
    feline_sequences
)

combined_metrics = calculate_metrics(
    combined_sequences
)


# ============================================================
# 9. VIRIDIS COLOR SCALE
# ============================================================

VIRIDIS_STOPS = [

    (0.00, "#440154"),

    (0.25, "#3B528B"),

    (0.50, "#21918C"),

    (0.75, "#5EC962"),

    (1.00, "#FDE725")

]


def hex_to_rgb(hex_color):

    hex_color = hex_color.lstrip("#")

    return (

        int(hex_color[0:2], 16),

        int(hex_color[2:4], 16),

        int(hex_color[4:6], 16)

    )


def rgb_to_hex(rgb):

    return "#{:02X}{:02X}{:02X}".format(

        int(rgb[0]),

        int(rgb[1]),

        int(rgb[2])

    )


def viridis_color(value):

    if value is None:

        return "#E0E0E0"

    value = max(
        0.0,
        min(1.0, value)
    )

    for i in range(
        len(VIRIDIS_STOPS) - 1
    ):

        p1, c1 = VIRIDIS_STOPS[i]

        p2, c2 = VIRIDIS_STOPS[i + 1]

        if p1 <= value <= p2:

            local_t = (
                (value - p1)
                / (p2 - p1)
            )

            rgb1 = hex_to_rgb(c1)

            rgb2 = hex_to_rgb(c2)

            rgb = tuple(

                rgb1[j]
                + (
                    rgb2[j]
                    - rgb1[j]
                ) * local_t

                for j in range(3)

            )

            return rgb_to_hex(rgb)

    return VIRIDIS_STOPS[-1][1]


# ============================================================
# 10. REFERENCE-BASED POSITION MAPPING
# ============================================================
#
#
# ============================================================

def reference_position_map(
        reference_sequence
    ):

    positions = []

    biological_position = 0

    for alignment_position in range(
        alignment_length
    ):

        residue = reference_sequence[
            alignment_position
        ].upper()

        # 
        if residue not in missing_chars:

            biological_position += 1

            positions.append(
                biological_position
            )

        else:

            # 
            positions.append(None)

    return positions


human_positions = reference_position_map(
    human_reference_sequence
)

feline_positions = reference_position_map(
    feline_reference_sequence
)

combined_positions = reference_position_map(
    combined_reference_sequence
)


# ============================================================
# 11. CREATE SVG HEATMAP
# ============================================================

def create_heatmap(

        group_name,

        metrics,

        reference_sequence,

        reference_name,

        reference_positions,

        n_sequences,

        output_file

    ):

    conservation = metrics[
        "conservation"
    ]

    entropy = metrics[
        "entropy"
    ]

    L = alignment_length

    # --------------------------------------------------------
    # Dimensions
    # --------------------------------------------------------

    cell_width = max(
        1.5,
        min(6, 1800 / L)
    )

    left_margin = 20

    right_margin = 210

    top_margin = 65

    heat_height = 32

    gap = 35

    bottom_margin = 75

    heat_width = (
        L * cell_width
    )

    svg_width = (
        left_margin
        + heat_width
        + right_margin
    )

    svg_height = (
        top_margin
        + heat_height
        + gap
        + heat_height
        + bottom_margin
    )

    svg = []

    # ========================================================
    # SVG START
    # ========================================================

    svg.append(

        f'<svg xmlns="http://www.w3.org/2000/svg" '
        f'width="{svg_width}" '
        f'height="{svg_height}" '
        f'viewBox="0 0 {svg_width} {svg_height}">'

    )

    svg.append(
        '<rect width="100%" '
        'height="100%" fill="white"/>'
    )

    # ========================================================
    # TITLE
    # ========================================================

    svg.append(

        f'<text '
        f'x="{left_margin}" '
        f'y="28" '
        f'font-family="Arial, Helvetica, sans-serif" '
        f'font-size="20" '
        f'font-weight="bold">'
        f'{escape(group_name)}'
        f'</text>'

    )

    svg.append(

        f'<text '
        f'x="{left_margin}" '
        f'y="48" '
        f'font-family="Arial, Helvetica, sans-serif" '
        f'font-size="12" '
        f'fill="#444444">'
        f'n = {n_sequences} sequences | '
        f'reference = {escape(reference_name)}'
        f'</text>'

    )

    # ========================================================
    # CONSERVATION HEATMAP
    # ========================================================

    y_cons = top_margin

    for i, value in enumerate(
        conservation
    ):

        x = (
            left_margin
            + i * cell_width
        )

        fill = viridis_color(
            value
        )

        svg.append(

            f'<rect '
            f'x="{x:.2f}" '
            f'y="{y_cons}" '
            f'width="{cell_width + 0.2:.2f}" '
            f'height="{heat_height}" '
            f'fill="{fill}" '
            f'stroke="none"/>'

        )

    # ========================================================
    # SHANNON ENTROPY HEATMAP
    # ========================================================

    y_entropy = (
        y_cons
        + heat_height
        + gap
    )

    for i, value in enumerate(
        entropy
    ):

        x = (
            left_margin
            + i * cell_width
        )

        if value is None:

            normalized = None

        else:

            normalized = (
                value
                / max_entropy
            )

        fill = viridis_color(
            normalized
        )

        svg.append(

            f'<rect '
            f'x="{x:.2f}" '
            f'y="{y_entropy}" '
            f'width="{cell_width + 0.2:.2f}" '
            f'height="{heat_height}" '
            f'fill="{fill}" '
            f'stroke="none"/>'

        )

    # ========================================================
    # X-AXIS
    # ========================================================

    axis_y = (
        y_entropy
        + heat_height
        + 20
    )

    svg.append(

        f'<line '
        f'x1="{left_margin}" '
        f'y1="{axis_y}" '
        f'x2="{left_margin + heat_width}" '
        f'y2="{axis_y}" '
        f'stroke="#333333" '
        f'stroke-width="1"/>'

    )

    # --------------------------------------------------------
    # Biological length = number of non-gap residues
    # in reference.
    # --------------------------------------------------------

    valid_reference_positions = [

        p for p in reference_positions

        if p is not None

    ]

    biological_length = max(
        valid_reference_positions
    )

    # Approximately 12 ticks
    tick_step = max(

        1,

        int(
            math.ceil(
                biological_length / 12
            )
        )

    )

    # Nice numbers
    if tick_step < 10:

        tick_step = 10

    elif tick_step < 50:

        tick_step = 25

    elif tick_step < 100:

        tick_step = 50

    else:

        tick_step = 100

    # --------------------------------------------------------
    # Put tick exactly where the reference reaches that
    # biological position.
    # --------------------------------------------------------

    for biological_position in range(

        tick_step,

        biological_length + 1,

        tick_step

    ):

        alignment_index = None

        for i, position in enumerate(
            reference_positions
        ):

            if position == biological_position:

                alignment_index = i

                break

        if alignment_index is None:

            continue

        x = (

            left_margin
            + alignment_index * cell_width
            + cell_width / 2

        )

        svg.append(

            f'<line '
            f'x1="{x:.2f}" '
            f'y1="{axis_y}" '
            f'x2="{x:.2f}" '
            f'y2="{axis_y + 6}" '
            f'stroke="#333333" '
            f'stroke-width="1"/>'

        )

        svg.append(

            f'<text '
            f'x="{x:.2f}" '
            f'y="{axis_y + 20}" '
            f'text-anchor="middle" '
            f'font-family="Arial, Helvetica, sans-serif" '
            f'font-size="10">'
            f'{biological_position}'
            f'</text>'

        )

    # ========================================================
    # CONSERVATION COLOR BAR
    # ========================================================

    legend_x = (

        left_margin
        + heat_width
        + 30

    )

    legend_width = 130

    legend_height = 14

    legend_y1 = y_cons + 8

    svg.append(

        f'<text '
        f'x="{legend_x}" '
        f'y="{legend_y1 - 7}" '
        f'font-family="Arial, Helvetica, sans-serif" '
        f'font-size="11" '
        f'font-weight="bold">'
        f'Conservation'
        f'</text>'

    )

    steps = 80

    for j in range(steps):

        t = j / (steps - 1)

        color = viridis_color(t)

        x = (
            legend_x
            + t * legend_width
        )

        svg.append(

            f'<rect '
            f'x="{x:.2f}" '
            f'y="{legend_y1}" '
            f'width="{legend_width / steps + 1:.2f}" '
            f'height="{legend_height}" '
            f'fill="{color}" '
            f'stroke="none"/>'

        )

    for value in [0, 0.5, 1]:

        x = (
            legend_x
            + value * legend_width
        )

        svg.append(

            f'<line '
            f'x1="{x:.2f}" '
            f'y1="{legend_y1 + legend_height}" '
            f'x2="{x:.2f}" '
            f'y2="{legend_y1 + legend_height + 4}" '
            f'stroke="#333333"/>'

        )

        svg.append(

            f'<text '
            f'x="{x:.2f}" '
            f'y="{legend_y1 + legend_height + 17}" '
            f'text-anchor="middle" '
            f'font-family="Arial, Helvetica, sans-serif" '
            f'font-size="10">'
            f'{value:.1f}'
            f'</text>'

        )

    # ========================================================
    # SHANNON ENTROPY COLOR BAR
    # ========================================================

    legend_y2 = y_entropy + 8

    svg.append(

        f'<text '
        f'x="{legend_x}" '
        f'y="{legend_y2 - 7}" '
        f'font-family="Arial, Helvetica, sans-serif" '
        f'font-size="11" '
        f'font-weight="bold">'
        f'Shannon entropy (bits)'
        f'</text>'

    )

    for j in range(steps):

        t = j / (steps - 1)

        color = viridis_color(t)

        x = (
            legend_x
            + t * legend_width
        )

        svg.append(

            f'<rect '
            f'x="{x:.2f}" '
            f'y="{legend_y2}" '
            f'width="{legend_width / steps + 1:.2f}" '
            f'height="{legend_height}" '
            f'fill="{color}" '
            f'stroke="none"/>'

        )

    entropy_values = [

        0,

        max_entropy / 2,

        max_entropy

    ]

    for value in entropy_values:

        normalized = (
            value
            / max_entropy
        )

        x = (
            legend_x
            + normalized * legend_width
        )

        svg.append(

            f'<line '
            f'x1="{x:.2f}" '
            f'y1="{legend_y2 + legend_height}" '
            f'x2="{x:.2f}" '
            f'y2="{legend_y2 + legend_height + 4}" '
            f'stroke="#333333"/>'

        )

        svg.append(

            f'<text '
            f'x="{x:.2f}" '
            f'y="{legend_y2 + legend_height + 17}" '
            f'text-anchor="middle" '
            f'font-family="Arial, Helvetica, sans-serif" '
            f'font-size="10">'
            f'{value:.2f}'
            f'</text>'

        )

    # ========================================================
    # FOOTNOTE
    # ========================================================

    explanation_y = svg_height - 15

    svg.append(

        f'<text '
        f'x="{left_margin}" '
        f'y="{explanation_y}" '
        f'font-family="Arial, Helvetica, sans-serif" '
        f'font-size="10" '
        f'fill="#555555">'
        f'X-axis = reference-based biological position '
        f'(gaps in reference not counted)'
        f'</text>'

    )

    svg.append("</svg>")

    # ========================================================
    # SAVE
    # ========================================================

    with open(
        output_file,
        "w",
        encoding="utf-8"
    ) as f:

        f.write(
            "\n".join(svg)
        )


# ============================================================
# 12. CREATE THREE HEATMAPS
# ============================================================

create_heatmap(

    "Human papillomavirus",

    human_metrics,

    human_reference_sequence,

    human_reference_name,

    human_positions,

    len(human_sequences),

    os.path.join(

        output_dir,

        "Human_conservation_entropy_heatmap.svg"

    )

)


create_heatmap(

    "Feline papillomavirus",

    feline_metrics,

    feline_reference_sequence,

    feline_reference_name,

    feline_positions,

    len(feline_sequences),

    os.path.join(

        output_dir,

        "Feline_conservation_entropy_heatmap.svg"

    )

)


create_heatmap(

    "Human + Feline papillomavirus",

    combined_metrics,

    combined_reference_sequence,

    combined_reference_name,

    combined_positions,

    len(combined_sequences),

    os.path.join(

        output_dir,

        "Human_Feline_combined_conservation_entropy_heatmap.svg"

    )

)


# ============================================================
# 13. SAVE NUMERICAL RESULTS
# ============================================================

csv_file = os.path.join(

    output_dir,

    "Conservation_Shannon_entropy_results.csv"

)


with open(

    csv_file,

    "w",

    newline="",

    encoding="utf-8-sig"

) as f:

    writer = csv.writer(f)

    writer.writerow([

        "Alignment_Column",

        "Human_Reference_Position",
        "Human_Consensus",
        "Human_Valid_N",
        "Human_Conservation",
        "Human_Shannon_Entropy",

        "Feline_Reference_Position",
        "Feline_Consensus",
        "Feline_Valid_N",
        "Feline_Conservation",
        "Feline_Shannon_Entropy",

        "Combined_Reference_Position",
        "Combined_Consensus",
        "Combined_Valid_N",
        "Combined_Conservation",
        "Combined_Shannon_Entropy"

    ])


    for i in range(
        alignment_length
    ):

        writer.writerow([

            i + 1,

            human_positions[i],

            human_metrics[
                "consensus"
            ][i],

            human_metrics[
                "valid_count"
            ][i],

            human_metrics[
                "conservation"
            ][i],

            human_metrics[
                "entropy"
            ][i],


            feline_positions[i],

            feline_metrics[
                "consensus"
            ][i],

            feline_metrics[
                "valid_count"
            ][i],

            feline_metrics[
                "conservation"
            ][i],

            feline_metrics[
                "entropy"
            ][i],


            combined_positions[i],

            combined_metrics[
                "consensus"
            ][i],

            combined_metrics[
                "valid_count"
            ][i],

            combined_metrics[
                "conservation"
            ][i],

            combined_metrics[
                "entropy"
            ][i]

        ])


# ============================================================
# 14. SUMMARY
# ============================================================

def mean(values):

    valid = [

        x for x in values

        if x is not None

    ]

    if not valid:

        return None

    return sum(valid) / len(valid)


print("\n==============================================")
print("SUMMARY")
print("==============================================")


for name, metrics, positions, reference_name in [

    (
        "Human",
        human_metrics,
        human_positions,
        human_reference_name
    ),

    (
        "Feline",
        feline_metrics,
        feline_positions,
        feline_reference_name
    ),

    (
        "Combined",
        combined_metrics,
        combined_positions,
        combined_reference_name
    )

]:

    valid_positions = [

        p for p in positions

        if p is not None

    ]

    biological_length = max(
        valid_positions
    )

    print(
        f"\n{name}"
    )

    print(
        f"Reference : "
        f"{reference_name}"
    )

    print(
        f"Gap-free reference length : "
        f"{biological_length}"
    )

    print(
        f"Mean conservation : "
        f"{mean(metrics['conservation']):.4f}"
    )

    print(
        f"Mean Shannon entropy : "
        f"{mean(metrics['entropy']):.4f} bits"
    )


print("\n==============================================")
print("FILES CREATED")
print("==============================================")


print(
    os.path.join(
        output_dir,
        "Human_conservation_entropy_heatmap.svg"
    )
)


print(
    os.path.join(
        output_dir,
        "Feline_conservation_entropy_heatmap.svg"
    )
)


print(
    os.path.join(
        output_dir,
        "Human_Feline_combined_conservation_entropy_heatmap.svg"
    )
)


print(csv_file)


print("\nDone.")